In [247]:
import torch
import torch.nn as nn
import numpy as np

def im2col(X, kernel_shape, stride=1, padding=(0, 0)):
    B, C = X.shape[:2]
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    L = out_H * out_W

    cols = np.zeros((B, C * kH * kW, L))

    patch_idx = 0
    for i in range(out_H):
        for j in range(out_W):

            h_start = i * stride
            w_start = j * stride

            # slice patch for ALL channels
            patch = X_padded[:, :, h_start:h_start + kH, w_start:w_start + kW]

            # flatten channels + kernel dims
            patch = patch.reshape(B, -1)    # → (B, C*kH*kW)

            cols[:, :, patch_idx] = patch
            patch_idx += 1
    
    return cols, out_H, out_W

def col2im(cols, output_shape, kernel_shape, stride=1, padding=0):
    B, C_k, num_cols = cols.shape   # C_k = C * kH * kW
    H, W = output_shape
    kH, kW = kernel_shape

    # padded spatial dims
    H_p, W_p = H + 2*padding, W + 2*padding

    # output tensor
    X_padded = np.zeros((B, C_k // (kH*kW), H_p, W_p))

    # how many sliding positions?
    out_H = (H_p - kH) // stride + 1
    out_W = (W_p - kW) // stride + 1

    idx = 0
    for i in range(out_H):
        for j in range(out_W):
            h_start = i * stride
            w_start = j * stride

            # reshape column back into (B, C, kH, kW)
            patch = cols[:, :, idx].reshape(B, -1, kH, kW)

            # scatter-add patch into spatial tensor
            X_padded[:, :, h_start:h_start+kH, w_start:w_start+kW] += patch

            idx += 1

    # remove padding
    if padding > 0:
        X_padded = X_padded[:, :, padding:-padding, padding:-padding]

    return X_padded

def conv2d_im2col_multi(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    B = X.shape[0]
    X_col, out_H, out_W = im2col(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)

    print(f'x: {X_col.shape}, W: {W_col.shape}')

    Y_col = W_col @ X_col

    Y = Y_col.reshape(B, C_out, out_H, out_W)
    return Y

def conv_transpose2d_img2col_multi(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    B, C, _, _ = Y.shape
    Y_col = Y.reshape(B, C, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[2]-1) * stride - 2*padding + kH
        W_out = (Y.shape[3]-1) * stride - 2*padding + kW
        output_shape = (H_out, W_out)

    X = col2im(X_col, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

In [248]:
#torch.manual_seed(1)

B = 30
C_out, C_in = 9, 5
input_size_H = 28
input_size_W = 35
kernel_size=6

stride=7
padding=12

x = torch.randn(B, C_in, input_size_H, input_size_W)

print(f'x: {x.shape}')

x: torch.Size([30, 5, 28, 35])


In [249]:
conv = nn.Conv2d(C_in, C_out, kernel_size=kernel_size, stride=stride, padding=padding)
convt = nn.ConvTranspose2d(C_out, C_in, kernel_size=kernel_size, stride=stride, padding=padding)

with torch.no_grad():
    #onv.weight.copy_(kernel)
    conv.bias.zero_()
    #convt.weight.copy_(kernel)
    convt.bias.zero_()

output_conv = conv(x)
print(f'Conv out: {output_conv.shape}')

output_convt = convt(output_conv)
print(f'ConvT out: {output_convt.shape}')

Conv out: torch.Size([30, 9, 7, 8])
ConvT out: torch.Size([30, 5, 24, 31])


In [250]:
x = np.array(x)
kernel1 = conv.weight.detach().numpy()
kernel2 = convt.weight.detach().numpy()

print(f'kernel shapes: {kernel1.shape}, {kernel2.shape}')

kernel shapes: (9, 5, 6, 6), (9, 5, 6, 6)


/tmp/ipykernel_3978/1592921002.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  x = np.array(x)


In [251]:
result = conv2d_im2col_multi(x, kernel1, stride=stride, padding=padding)
print(f'result: {result.shape}')

x: (30, 180, 56), W: (9, 180)
result: (30, 9, 7, 8)


In [252]:
original = conv_transpose2d_img2col_multi(result, kernel2, stride=stride, padding=padding, output_shape=None)
print(f'original: {original.shape}')

original: (30, 5, 24, 31)


In [ ]:
print(np.allclose(output_conv.detach().numpy(), result, atol=1e-6, rtol=1e-6))
print(np.max(np.abs(output_conv.detach().numpy() - result)))

True
8.318662139128463e-07


In [254]:
print(np.allclose(output_convt.detach().numpy(), original, atol=1e-6, rtol=1e-6))
print(np.max(np.abs(output_convt.detach().numpy() - original)))

True
9.73021528627438e-08
